In [1]:
import requests 
import pandas as pd 
import numpy as np
import sys 
import os
import geopandas as gpd
import json
from scipy.spatial import cKDTree
import importlib
import dist_func

importlib.reload(dist_func)
from dist_func import nearest_neighbor, find_line_diversity, combine_connectivity_score

## Loading London underground station location data

In [2]:
url = f"https://services1.arcgis.com/YswvgzOodUvqkoCN/ArcGIS/rest/services/TfL_stations/FeatureServer/1/query"

batch_size = 2000
offset = 0

all_features = []
res = []
while True:
    params = {
    "where": "1=1",
    "outFields": "*",
    "returnGeometry": True,
    "outSR": 4326,  
    "f": "geojson",
    "resultOffset": offset,
    "resultRecordCount": batch_size
}
    response = requests.get(url, params=params).json()
    res.extend([{k:v} for k,v in response.items()])
    #print(response)
    all_features.extend(response['features'])

    if not response['features']:
        break

    offset += batch_size

print("Total records:", len(all_features))

# Export to GeoJSON file

geojson_data = {
    "type": "FeatureCollection",
    "features": all_features
}

with open("tfl_stations.geojson", 'w', encoding='utf-8') as f:
    json.dump(geojson_data, f, ensure_ascii=False, indent=2)

print(f"Exported to tfl_stations.geojson")

Total records: 273
Exported to tfl_stations.geojson


In [3]:
# Check structure and duplicates
print(f"Total features: {len(all_features)}")
print(f"\nFirst feature keys: {all_features[0].keys()}")
print(f"Sample feature:\n{all_features[0]}\n")

# Get station names (GeoJSON format uses 'properties')
station_names = [f['properties']['NAME'] for f in all_features]
station_ids = [f['properties']['OBJECTID'] for f in all_features]

print(f"Unique station names: {len(set(station_names))}")
print(f"Unique OBJECTID: {len(set(station_ids))}")

# Find duplicate names
from collections import Counter
name_counts = Counter(station_names)
duplicates = {name: count for name, count in name_counts.items() if count > 1}

if duplicates:
    print(f"\nDuplicate station names:")
    for name, count in duplicates.items():
        print(f"  {name}: appears {count} times")
        # Show the duplicate entries
        for f in all_features:
            if f['properties']['NAME'] == name:
                print(f"    OBJECTID: {f['properties']['OBJECTID']}, Lines: {f['properties']['LINES']}")

Total features: 273

First feature keys: dict_keys(['type', 'id', 'geometry', 'properties'])
Sample feature:
{'type': 'Feature', 'id': 111, 'geometry': {'type': 'Point', 'coordinates': [-0.0974918448345559, 51.5149545361712]}, 'properties': {'OBJECTID': 111, 'NAME': "St. Paul's", 'LINES': 'Central', 'ATCOCODE': '940GZZLUSPU', 'MODES': 'bus, tube', 'ACCESSIBILITY': 'Not Accessible', 'NIGHT_TUBE': 'Yes', 'NETWORK': 'London Underground', 'DATASET_LAST_UPDATED': 1638144000000, 'FULL_NAME': "St. Paul's station"}}

Unique station names: 272
Unique OBJECTID: 273

Duplicate station names:
  Paddington: appears 2 times
    OBJECTID: 125, Lines: Hammersmith & City, District, Circle, Bakerloo
    OBJECTID: 171, Lines: Hammersmith & City, District, Circle, Bakerloo


In [4]:
gdf = gpd.read_file("tfl_stations.geojson")
print(f"Loaded {len(gdf)} stations")
gdf.head()

Loaded 273 stations


,OBJECTID,NAME,LINES,ATCOCODE,MODES,ACCESSIBILITY,NIGHT_TUBE,NETWORK,DATASET_LAST_UPDATED,FULL_NAME,geometry
0,111,St. Paul's,Central,940GZZLUSPU,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,St. Paul's station,POINT (-0.09749 51.51495)
1,112,Mile End,"District, Hammersmith & City, Central",940GZZLUMED,"bus, tube",Partially Accessible - Interchange Only,Yes,London Underground,1638144000000,Mile End station,POINT (-0.03374 51.52524)
2,113,Bethnal Green,Central,940GZZLUBLG,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Bethnal Green station,POINT (-0.05466 51.52724)
3,114,Leyton,Central,940GZZLULYN,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Leyton station,POINT (-0.00546 51.55661)
4,115,Snaresbrook,Central,940GZZLUSNB,"bus, tube",Not Accessible,Yes,London Underground,1638144000000,Snaresbrook station,POINT (0.02142 51.58068)


In [5]:
import importlib
sys.path.append(os.path.abspath("../"))

# Force reload to pick up latest version of the module
if 'dist_func' in sys.modules:
    importlib.reload(sys.modules['dist_func'])
from dist_func import nearest_neighbor

In [6]:
grids = pd.read_csv("../Demand Prediction Model/Output/sorted_grid.csv")

In [7]:
grids.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2464 entries, 0 to 2463
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   geometry       2464 non-null   object 
 1   gravity_score  2464 non-null   float64
 2   gravity_log    2464 non-null   float64
 3   lat            2464 non-null   float64
 4   lon            2464 non-null   float64
dtypes: float64(4), object(1)
memory usage: 96.4+ KB


### All in one cell

In [ ]:
result = nearest_neighbor(grids, gdf, k=3)
result = find_line_diversity(result, gdf, k=3)
result = combine_connectivity_score(result, alpha=0.5, beta=0.5)

cols = [
    'harmonic_mean_adj_dist_norm',
    'line_unique_count_norm',
    'diversity_penalty',
    'connectivity_score'
]

print('Computed columns present:', [c for c in cols if c in result.columns])
print('connectivity_score range:', result['connectivity_score'].min(), 'to', result['connectivity_score'].max())

result.head()

Total 2464 grid points
Mapping 273 stations
Computed columns present: ['harmonic_mean_adj_dist_norm', 'line_unique_count_norm', 'diversity_penalty', 'connectivity_score']
connectivity_score range: 0.004705158694626762 to 1.0


,lon,lat,nearest_station_1,line_unique_count,connectivity_score
0,-0.412511,51.514517,South Ruislip,2,0.483048
1,-0.145405,51.348731,Morden,1,0.638856
2,-0.050901,51.545036,Bethnal Green,3,0.338494
3,-0.051669,51.527064,Bethnal Green,3,0.307885
4,-0.328057,51.459390,Richmond,2,0.452908


In [ ]:
output_cols = [
    'lon', 'lat',
    'nearest_station_1', 'nearest_station_2', 'nearest_station_3',
    'nearest_station_1_dist', 'nearest_station_2_dist', 'nearest_station_3_dist',
    'harmonic_mean_adj_dist', 'harmonic_mean_adj_dist_norm',
    'line_unique_count', 'line_unique_count_norm',
    'diversity_penalty',
    'connectivity_score'
]

# Keep only columns that exist (in case some optional ones are missing)
save_cols = [c for c in output_cols if c in result.columns]
connectivity_output = result[save_cols].copy()

# Save to CSV
output_path = "connectivity_scores.csv"
connectivity_output.to_csv(output_path, index=False)

print(f"Saved {len(connectivity_output)} rows to {output_path}")
print(f"Columns: {save_cols}")
print(f"\nconnectivity_score range: {connectivity_output['connectivity_score'].min():.4f} to {connectivity_output['connectivity_score'].max():.4f}")
print(connectivity_output.head())

### Validation
1. Cell 1: City-wise score map
2. Cell 2: Single grid point with the highest connectivity gap

In [ ]:
import folium
import branca.colormap as cm
import numpy as np
import os

m_all = folium.Map(location=[51.5074, -0.1278], zoom_start=10, tiles="CartoDB positron")

vmin = float(result["connectivity_score"].min())
vmax = float(result["connectivity_score"].max())
cmap = cm.LinearColormap(
    ["#2c7bb6", "#abd9e9", "#ffffbf", "#fdae61", "#d7191c"],
    vmin=vmin,
    vmax=vmax
)
cmap.caption = "Connectivity Score"

for _, r in result.iterrows():
    folium.CircleMarker(
        location=[float(r["lat"]), float(r["lon"])],
        radius=2.5,
        color=cmap(float(r["connectivity_score"])),
        fill=True,
        fill_opacity=0.75,
        weight=0,
        popup=(
            f"score={r['connectivity_score']:.3f}<br>"
            f"line_unique={r.get('line_unique_count', np.nan)}<br>"
            f"nearest={r.get('nearest_station_1', 'NA')}"
        )
    ).add_to(m_all)

cmap.add_to(m_all)

output_html = os.path.abspath("connectivity_citywide_map.html")
m_all.save(output_html)
print(f"Saved map to: {output_html}")

m_all

Saved map to: c:\Users\harry\Desktop\sig_datathon\Connectivity\connectivity_citywide_map.html


In [18]:
# Single-grid drill-down: inspect one grid point and its 3 nearest stations
def drill_down_grid(result_df, stations_gdf, grid_index=None, lon=None, lat=None, zoom_start=13, save_html=True):
    required_cols = [
        "lon", "lat", "connectivity_score",
        "nearest_station_1", "nearest_station_2", "nearest_station_3"
    ]
    missing = [c for c in required_cols if c not in result_df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if grid_index is None:
        if lon is None or lat is None:
            raise ValueError("Provide grid_index, or both lon and lat.")
        d2 = (result_df["lon"] - float(lon)) ** 2 + (result_df["lat"] - float(lat)) ** 2
        grid_index = int(d2.idxmin())

    row = result_df.loc[grid_index]
    center = [float(row["lat"]), float(row["lon"]) ]

    # Ensure stations are map-ready for folium (lat/lon)
    if getattr(stations_gdf, "crs", None) is not None and stations_gdf.crs.to_epsg() != 4326:
        stations_map = stations_gdf.to_crs(epsg=4326)
    else:
        stations_map = stations_gdf

    m = folium.Map(location=center, zoom_start=zoom_start, tiles="CartoDB positron")

    grid_popup = (
        f"grid_index={grid_index}<br>"
        f"score={row['connectivity_score']:.4f}<br>"
        f"line_unique_count={row.get('line_unique_count', np.nan)}<br>"
        f"harmonic_mean_adj_dist={row.get('harmonic_mean_adj_dist', np.nan):.4f}"
    )

    folium.CircleMarker(
        location=center,
        radius=8,
        color="red",
        fill=True,
        fill_opacity=0.95,
        popup=folium.Popup(grid_popup, max_width=300)
    ).add_to(m)

    for i in [1, 2, 3]:
        station_col = f"nearest_station_{i}"
        station_name = row.get(station_col)
        if pd.isna(station_name):
            continue

        station_match = stations_map[stations_map["NAME"] == station_name]
        if station_match.empty:
            continue

        station_pt = station_match.iloc[0].geometry
        s_lat = float(station_pt.y)
        s_lon = float(station_pt.x)
        dist_col = f"nearest_station_{i}_dist"
        dist_val = row.get(dist_col, np.nan)

        folium.Marker(
            location=[s_lat, s_lon],
            popup=f"{i}. {station_name}<br>adj_dist={dist_val:.4f}",
            icon=folium.Icon(color="blue", icon="info-sign")
        ).add_to(m)

        folium.PolyLine(
            locations=[center, [s_lat, s_lon]],
            color="gray",
            weight=2,
            opacity=0.8
        ).add_to(m)

    if save_html:
        single_html = os.path.abspath(f"connectivity_single_grid_{grid_index}.html")
        m.save(single_html)
        print(f"Saved single-grid map to: {single_html}")

    return m

# Example: drill into the highest-scoring grid
top_idx = int(result["connectivity_score"].idxmax())
m_single = drill_down_grid(result, gdf, grid_index=top_idx, zoom_start=13, save_html=True)
m_single

Saved single-grid map to: c:\Users\harry\Desktop\sig_datathon\Connectivity\connectivity_single_grid_2443.html
